In [1]:
!pip --version

pip 26.2.1 from D:\in0902\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [4]:
llm = ChatOpenAI(temperature=0, model="gpt-4.1-mini")
llm.invoke("호주의 수도는 뭐야")

AIMessage(content='호주의 수도는 캔버라(Canberra)입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 13, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_27fa8c7d29', 'id': 'chatcmpl-EOYdh9k5eEFzBDDA1Dv9NaNJsbM7u', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a7ce-a5c7-7df2-ad90-156b1974b2c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 14, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
class EmailSummary(BaseModel):
    person:str = Field (description= "메일을 보낸 사람")
    email:str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject:str =Field(description="메일 제목")
    summary:str= Field(description="메일 본문을 요약한 텍스트")
    date:str =Field(description="메일 본문에 업급된 미팅 날짜와 시간")

    
    

In [6]:
# 이메일 예시
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [7]:
#네이티브 API 방식 : 모델에 스키마 강제
llm_with_structered = ChatOpenAI(
    temperature=0, model="gpt-4.1-mini"
).with_structured_output(EmailSummary)

In [8]:
answer = llm_with_structered.invoke(email_conversation)
answer

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='김철수 상무가 이은채 대리에게 ZENESIS 자전거에 대한 상세 브로슈어 요청과 함께, 기술 사양, 배터리 성능, 디자인 정보가 필요하다고 전달했습니다. 또한, 1월 15일 화요일 오전 10시에 미팅을 제안하며 협력 가능성을 논의하고자 합니다.', date='2024-01-08')

In [9]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

In [17]:
# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()


#출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()

In [18]:
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [19]:
# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    # 주제에 대한 다섯가지(영어)를 나열하라는 템플릿
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],  # 입력 변수로 'subject' 사용
    # 부분 변수로 형식 지침 사용
    partial_variables={"format_instructions": format_instructions},
)

In [20]:
# 프롬프트2 템플릿 설정
prompt2 = PromptTemplate(
    # 주제에 대한 다섯가지(한글)를 나열하라는 템플릿
    template="다섯가지 {subject}.\n{format_instructions}",
    input_variables=["subject"],  # 입력 변수로 'subject' 사용
    # 부분 변수로 형식 지침 사용
    partial_variables={"format_instructions": format_instructions},
)

In [21]:
# ChatOpenAI 모델 초기화
model = ChatOpenAI(temperature=0)

#프롬프트, 모델 , 출력 파서를 연결하여 체인 생성
chain = prompt | model | output_parser

In [22]:
#"일본 관광 명소"에 대한 체인 호출 실행
chain.invoke({"subject": "일본 관광명소"})

['도쿄 디즈니랜드', '후지산', '교토', '나라', '오사카']

In [23]:
#ChatOpenAI 모델 초기화
model = ChatOpenAI(temperature=0)

# 프롬프트 , 모델 , 출력 파서를 연결하여 ㅊ인 생성
chain2 = prompt2 | model | output_parser

#"인도 관광명소"에 대한 체인 호출 실행
chain2.invoke({"subject": "인도 관광명소"})

['타지마할', '자이푸르', '고아', '코찌칼', '바라나시']

In [24]:
#스트림을 순회.
for s in chain.stream({"subject": "대한민국 관광명소"}):
    print(s)

['경복궁']
['남산타워']
['부산 해운대해수욕장']
['제주도 성산일출봉']
['경주 불국사temples']
